# 03 交通事故点位排查

为交通安全调查设计一份现场排查候选清单。先审查事故坐标质量，再比较网格大小、行政区和排序口径对结果的影响。清单用于提出调查问题，不能直接当作治理效果或风险排名。

建议工作量：8—12小时。本工作本是项目起点，默认程序的输出不是完整作业答案。

## 最终成果
- 坐标质量与排除记录统计
- 网格事故分布与候选清单
- 尺度敏感性对照
- 现场核查问题与证据边界


## 数据与范围

[NYC Motor Vehicle Collisions - Crashes](https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/h9gi-nx95)

遵循 NYC Open Data 原始使用条款

完整请求2024年1月记录并核对API总数。工作台仅对通过边界检查的坐标聚合，保留被排除记录的数量。

- 事故记录数量没有交通暴露量分母。
- 坐标缺失会影响空间分布。
- 500米网格不是实际道路或社区边界。
- 伤者人数排序与事故次数排序回答不同问题。


In [ ]:
from pathlib import Path
import sys, json
candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in candidates if (p / "python" / "analyze.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the project-kit folder or its notebooks folder")
sys.path.insert(0, str(ROOT / "python"))
from analyze import run
OUTPUT = ROOT / "outputs" / "student"
OUTPUT.mkdir(parents=True, exist_ok=True)


## 参考分析与口径检查

先运行一次，解释每个指标的分母和单位。打开源码确认数据筛选规则。预测项目这里读取冻结模型的结果，不重新训练。


In [ ]:
project = "hotspots"
config = {
  "borough": "all",
  "cell": "500",
  "weight": "count"
}
result = run(project, config, ROOT / "data")
print(json.dumps(result["metrics"], ensure_ascii=False, indent=2))
print("Source SHA-256:", result["sourceSha"])


## 对照实验

以下配置提供一个可运行起点。说明每次只改变了什么，以及还存在哪些混杂条件。增加你自己的对照，不只重复默认结果。


In [ ]:
comparisons = [
  {
    "cell": "250",
    "weight": "count"
  },
  {
    "cell": "500",
    "weight": "count"
  },
  {
    "cell": "500",
    "weight": "injured"
  }
]
experiments = []
for change in comparisons:
    trial = run(project, {**config, **change}, ROOT / "data")
    experiments.append({"project": project, "config": trial["config"], "metrics": trial["metrics"], "sourceSha": trial["sourceSha"]})
    print(json.dumps(experiments[-1], ensure_ascii=False))
(OUTPUT / (project + "-comparison.json")).write_text(json.dumps(experiments, ensure_ascii=False, indent=2), encoding="utf-8")


## 01 任务界定

**限定排查对象**

明确月份、行政区与排序目标。为什么应称为“候选点位”而不是“最危险路段”？

阶段成果：研究范围和后续现场核查目标。

### 我的证据与解释

在这里填写自己的分析，引用结果行、实验参数或图表。


## 02 坐标审计

**量化无法落图的记录**

比较筛选前记录数和有效坐标数。说明这些排除是否可能改变空间判断。

阶段成果：坐标完整性表及可疑点处理规则。

### 我的证据与解释

在这里填写自己的分析，引用结果行、实验参数或图表。


## 03 空间对照

**比较空间尺度与权重**

至少比较250米、500米、1000米中的两种尺度，再对比事故次数与伤者人数排序。

阶段成果：2种尺度、2种权重的清单及变化说明。

### 我的证据与解释

在这里填写自己的分析，引用结果行、实验参数或图表。


## 04 排查建议

**将图表转为调查问题**

选择3个候选网格，说明需补充的道路结构、流量、暴露量或现场记录。

阶段成果：候选清单附坐标、选择理由、所缺证据，不写未经验证的治理结论。

### 我的证据与解释

在这里填写自己的分析，引用结果行、实验参数或图表。


## 深入分析

- 使用Python的DBSCAN对照网格方法，并检验距离阈值。
- 引入合规获取的道路长度或流量，讨论暴露标准化和时间覆盖差异。

项目包还包含 extensions.py 的可运行扩展。先安装 python/requirements.txt 中的依赖，再运行下面的命令。


In [ ]:
import subprocess
subprocess.run([sys.executable, str(ROOT / "python" / "extensions.py"), "--project", "hotspots", "--output", str(OUTPUT / "extensions")], check=True)


## 提交前自查

- [ ] 报告缺失坐标及行政区未知记录。
- [ ] 网格原点和尺度可复现。
- [ ] 至少一次敏感性分析，不把数量直接解释为风险。

报告应包含研究问题、方法对照、发现、局限、源数据哈希和复现命令。请附代码、配置、结果CSV。阶段文字与实验次数不自动换算成绩。
